# Notebook 02 : traitement distribué avec PySpark

TP2 BigData Commerce Analytics (B2 Sup de Vinci).

Ce notebook lit dans MongoDB les collections remplies par le notebook 01 (`raw_commandes` et `raw_payments`), les traite avec Spark, puis calcule les indicateurs du projet. Il faut donc avoir exécuté le notebook 01 avant.

## Atelier 2 (1/2) : préparer PySpark

On crée une session Spark en mode local avec `local[*]` : Spark tourne sur ce seul PC mais répartit le travail sur tous ses cœurs, comme un petit cluster. Spark ne sait pas lire MongoDB tout seul, on lui ajoute donc le connecteur officiel `mongo-spark-connector_2.13:11.1.0` (la version 11 est celle qui supporte Spark 4, qui utilise Scala 2.13).

Dans le code :
- les quatre lignes `os.environ` sont des adaptations à Windows (le TP est prévu sous Ubuntu) : Spark doit utiliser le Python du kernel, le nom du PC contient un accent qui casse l'adresse interne de Spark (`SPARK_LOCAL_HOSTNAME`), et Hadoop a besoin de `winutils.exe` et `hadoop.dll` (`HADOOP_HOME`) pour charger le connecteur ;
- `spark.jars.packages` télécharge le connecteur au premier lancement (1 à 2 minutes) ;
- `spark.sql.session.timeZone = UTC` affiche les dates comme elles sont stockées dans MongoDB, sans décalage d'une heure ;
- les `print` affichent les versions, le mode, le nombre de cœurs utilisés et l'adresse de Spark UI.

In [1]:
import os
import sys
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable                  # Windows : Spark utilise le Python du kernel
os.environ.setdefault("SPARK_LOCAL_HOSTNAME", "localhost")      # Windows : nom du PC avec accent
os.environ["HADOOP_HOME"] = r"C:\hadoop"                        # Windows : winutils.exe
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]     # Windows : hadoop.dll

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("BigDataCommerceAnalytics")
    .master("local[*]")
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0")
    .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Mode    :", spark.sparkContext.master)
print("Cœurs utilisés (parallélisme par défaut) :", spark.sparkContext.defaultParallelism)
print("Spark UI :", spark.sparkContext.uiWebUrl)

PySpark : 4.0.1
Spark   : 4.0.1
Mode    : local[*]
Cœurs utilisés (parallélisme par défaut) : 16
Spark UI : http://localhost:4040


## Atelier 2 (2/2) : premier traitement avec PySpark

### 1. Lire les données depuis MongoDB

Dans le code :
- `lire_collection(nom)` renvoie un DataFrame Spark à partir d'une collection de la base `commerce`, grâce au format `mongodb` apporté par le connecteur ;
- `printSchema()` affiche les colonnes et leur type, `show(5)` les 5 premières lignes, `count()` le nombre de lignes lues.

In [2]:
def lire_collection(nom):
    return (
        spark.read.format("mongodb")
        .option("database", "commerce")
        .option("collection", nom)
        .load()
    )


df_commandes = lire_collection("raw_commandes")
df_payments = lire_collection("raw_payments")

print("=== raw_commandes ===")
df_commandes.printSchema()
df_commandes.show(5, truncate=False)

print("=== raw_payments ===")
df_payments.printSchema()
df_payments.show(5, truncate=False)

print("Documents lus : commandes =", df_commandes.count(), "| paiements =", df_payments.count())

=== raw_commandes ===
root
 |-- _id: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- cart_discounted_total: double (nullable = true)
 |-- cart_id: integer (nullable = true)
 |-- cart_total: double (nullable = true)
 |-- category: string (nullable = true)
 |-- discount_percentage: double (nullable = true)
 |-- discounted_line_total: double (nullable = true)
 |-- line_total: double (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- username: string (nullable = true)

+------------------------+--------------+---------------------+-------+----------+------------------+-------------------+---------------------+-----------------+-------------------+----------+----------+----------------------+--------+--------

**Observations**

- Spark lit 800 commandes et 208 paiements, le même volume que dans MongoDB : la lecture est complète.
- Le schéma n'est écrit nulle part, c'est le connecteur qui le déduit en lisant un échantillon de documents (entiers, décimaux, textes, dates).
- Le champ `_id` n'existe pas dans nos fichiers sources. C'est l'identifiant technique (ObjectId) que MongoDB ajoute automatiquement à chaque document inséré, le connecteur le convertit en texte. Il n'a pas de sens métier, on le retire des analyses avec `.drop("_id")`.

### 2. Transformations et actions

On travaille sur les lignes de commandes : on garde les colonnes utiles, on filtre les lignes avec au moins 2 articles, on calcule la remise de chaque ligne, puis on regroupe par catégorie.

Dans le code :
- la première partie ne fait que des **transformations** : `drop`, `select` (colonnes utiles), `filter` (lignes avec `quantity >= 2`), `withColumn` (nouvelle colonne `remise` = prix avant remise - prix après remise), `groupBy` + `agg` (nombre de lignes, d'articles, montant et remise par catégorie) et `orderBy` (tri par montant décroissant) ;
- la deuxième partie déclenche les calculs avec des **actions** : `show`, `count`, et `take(3)` qui ramène les 3 premières lignes du résultat dans Python.

In [3]:
# Transformations : rien n'est calculé ici
df_lignes = (
    df_commandes
    .drop("_id")
    .select("cart_id", "user_id", "category", "product_title",
            "quantity", "unit_price", "line_total", "discounted_line_total")
    .filter(F.col("quantity") >= 2)
    .withColumn("remise", F.round(F.col("line_total") - F.col("discounted_line_total"), 2))
)

df_par_categorie = (
    df_lignes
    .groupBy("category")
    .agg(
        F.count("*").alias("nb_lignes"),
        F.sum("quantity").alias("nb_articles"),
        F.round(F.sum("discounted_line_total"), 2).alias("montant"),
        F.round(F.sum("remise"), 2).alias("remise_totale"),
    )
    .orderBy(F.desc("montant"))
)

# Actions : c'est ici que Spark lit MongoDB et exécute le plan
df_lignes.show(5, truncate=False)
print("Lignes avec quantity >= 2 :", df_lignes.count())
df_par_categorie.show(10, truncate=False)
top3 = df_par_categorie.take(3)
print("Top 3 des catégories :", [ligne["category"] for ligne in top3])

+-------+-------+------------------+----------------------+--------+----------+-----------------+---------------------+------+
|cart_id|user_id|category          |product_title         |quantity|unit_price|line_total       |discounted_line_total|remise|
+-------+-------+------------------+----------------------+--------+----------+-----------------+---------------------+------+
|1      |1      |tops              |Blue Frock            |4       |29.99     |119.96           |105.41               |14.55 |
|1      |1      |motorcycle        |Generic Motorcycle    |3       |3999.99   |11999.97         |10547.97             |1452.0|
|1      |1      |smartphones       |iPhone 6              |3       |299.99    |899.97           |839.76               |60.21 |
|1      |1      |sports-accessories|Baseball Ball         |2       |8.99      |17.98            |17.67                |0.31  |
|2      |2      |mens-shirts       |Man Short Sleeve Shirt|5       |19.99     |99.94999999999999|93.12         

| Type | Opérations utilisées | Rôle |
|---|---|---|
| Transformations | `drop`, `select`, `filter`, `withColumn`, `groupBy` + `agg`, `orderBy` | Décrivent un nouveau DataFrame à partir d'un autre. Rien n'est exécuté, Spark ajoute seulement l'opération à son plan. |
| Actions | `show`, `count`, `take` | Demandent un résultat (affichage, nombre, lignes ramenées dans Python). C'est à ce moment que Spark lit les données et exécute tout le plan. |

Une transformation renvoie un nouveau DataFrame, une action renvoie une valeur ou affiche un résultat.

### 3. Lazy evaluation

Pour cette partie, ouvrir Spark UI à côté du notebook (http://localhost:4040, onglet **Jobs**).

**Cellule A : transformations seules.** On enchaîne cinq transformations sur les paiements (garder les PAID, créer une colonne mois, regrouper, sommer, trier) sans aucune action. `explain()` affiche le plan que Spark a préparé, sans l'exécuter.

Après l'exécution : rafraîchir Spark UI, aucun nouveau job ne doit apparaître. Faire une capture d'écran (`captures/01_spark_ui_avant_action.png`).

In [13]:
nb_jobs_avant = len(spark.sparkContext.statusTracker().getJobIdsForGroup())

df_lazy = (
    df_payments
    .drop("_id")
    .filter(F.col("status") == "PAID")
    .withColumn("mois", F.date_format("payment_date", "yyyy-MM"))
    .groupBy("mois")
    .agg(F.round(F.sum("amount"), 2).alias("ca_encaisse"))
    .orderBy("mois")
)

print("DataFrame défini :", df_lazy)
df_lazy.explain()

nb_jobs_apres = len(spark.sparkContext.statusTracker().getJobIdsForGroup())
print("Jobs Spark lancés par cette cellule :", nb_jobs_apres - nb_jobs_avant)

DataFrame défini : DataFrame[mois: string, ca_encaisse: double]
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [mois#7967 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(mois#7967 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=2846]
      +- HashAggregate(keys=[mois#7967], functions=[sum(amount#18)])
         +- Exchange hashpartitioning(mois#7967, 200), ENSURE_REQUIREMENTS, [plan_id=2843]
            +- HashAggregate(keys=[mois#7967], functions=[partial_sum(amount#18)])
               +- Project [amount#18, date_format(payment_date#24, yyyy-MM, Some(UTC)) AS mois#7967]
                  +- Filter (isnotnull(status#26) AND (status#26 = PAID))
                     +- InMemoryTableScan [amount#18, payment_date#24, status#26], [isnotnull(status#26), (status#26 = PAID)]
                           +- InMemoryRelation [amount#18, amount_collected#19, cart_id#20, nb_articles#21, nb_lignes#22, order_id#23, payment_date#24, payment_id#25, status#26, user_id#27]

**Cellule B : l'action.** `show()` demande un résultat : Spark exécute maintenant tout le plan.

Après l'exécution : rafraîchir Spark UI, de nouveaux jobs sont apparus. Faire une capture (`captures/02_spark_ui_apres_action.png`), puis cliquer sur le dernier job pour voir ses Stages et ses Tasks.

In [14]:
nb_jobs_avant = len(spark.sparkContext.statusTracker().getJobIdsForGroup())

df_lazy.show()

nb_jobs_apres = len(spark.sparkContext.statusTracker().getJobIdsForGroup())
print("Jobs Spark lancés par cette cellule :", nb_jobs_apres - nb_jobs_avant)

+-------+-----------+
|   mois|ca_encaisse|
+-------+-----------+
|2025-01|   74046.27|
|2025-02|  432281.84|
|2025-03|  384654.44|
|2025-04|  228117.54|
|2025-05|   128046.3|
|2025-06|  321302.66|
|2025-07|   54081.45|
|2025-08|  140272.22|
|2025-09|  102087.86|
|2025-10|  203587.26|
|2025-11|  285127.81|
|2025-12|  315629.96|
+-------+-----------+

Jobs Spark lancés par cette cellule : 2


**Conclusion sur la lazy evaluation**

Après la cellule A, aucun job n'est lancé alors qu'on a défini cinq transformations : Spark a seulement construit un plan d'exécution, celui qu'affiche `explain()`. Les jobs n'apparaissent qu'avec la cellule B, au moment de l'action `show()`.

Spark exécute donc réellement les transformations seulement quand une action demande un résultat : c'est la lazy evaluation (évaluation paresseuse). L'intérêt est que Spark connaît tout le traitement avant de le lancer et peut l'optimiser. Dans le plan affiché par `explain()`, on voit par exemple qu'il ne lit que les colonnes dont il a besoin (`status`, `payment_date`, `amount`) au lieu de toute la collection.

### 4. Partitions et Spark UI

Un DataFrame Spark est découpé en **partitions**, des morceaux de données que Spark traite en parallèle.

Dans le code :
- `rdd.getNumPartitions()` donne le nombre de partitions du DataFrame ;
- `spark_partition_id()` indique dans quelle partition se trouve chaque ligne, on compte ensuite les lignes par partition ;
- `repartition(4)` redécoupe les commandes en 4 partitions pour comparer, puis on relance une agrégation pour l'observer dans Spark UI.

In [6]:
print("Partitions de raw_commandes :", df_commandes.rdd.getNumPartitions())
print("Partitions de raw_payments  :", df_payments.rdd.getNumPartitions())
(df_commandes
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show())

df_commandes_4 = df_commandes.repartition(4)
print("Après repartition(4) :", df_commandes_4.rdd.getNumPartitions(), "partitions")
(df_commandes_4
    .groupBy(F.spark_partition_id().alias("partition"))
    .count()
    .orderBy("partition")
    .show())

# Agrégation sur les 4 partitions, à observer dans Spark UI (Jobs > Stages > Tasks)
(df_commandes_4
    .groupBy("category")
    .agg(F.round(F.sum("discounted_line_total"), 2).alias("montant"))
    .orderBy(F.desc("montant"))
    .show(5))

Partitions de raw_commandes : 1
Partitions de raw_payments  : 1
+---------+-----+
|partition|count|
+---------+-----+
|        0|  800|
+---------+-----+

Après repartition(4) : 4 partitions
+---------+-----+
|partition|count|
+---------+-----+
|        0|  200|
|        1|  200|
|        2|  200|
|        3|  200|
+---------+-----+

+--------------+----------+
|      category|   montant|
+--------------+----------+
|       vehicle|1682064.05|
|    motorcycle| 609460.61|
|  mens-watches| 593063.86|
|womens-watches| 249189.51|
|       laptops|  85006.24|
+--------------+----------+
only showing top 5 rows


**Observations et lien entre partitions, tasks, stages et jobs**

Avec si peu de données (800 documents, moins d'un mégaoctet), le connecteur MongoDB ne crée qu'une seule partition : toute la collection est traitée par une seule task. Après `repartition(4)`, les 800 lignes sont réparties équitablement, 200 par partition.

- Une **task** est le traitement d'une partition par un cœur. 1 partition donne 1 task, 4 partitions donnent 4 tasks qui peuvent tourner en même temps.
- Un **stage** est un groupe de tasks qui s'enchaînent sans échanger de données entre partitions. Dès qu'une opération oblige à redistribuer les données (`groupBy`, `orderBy`, `repartition`), Spark fait un *shuffle* et commence un nouveau stage.
- Un **job** est l'ensemble des stages lancés par une action (`show`, `count`...).

Donc : une action → un ou plusieurs jobs → chaque job est découpé en stages aux endroits où il y a un shuffle → chaque stage lance une task par partition.

**Ce qu'on regarde dans Spark UI (http://localhost:4040)**

| Onglet | Ce qu'on y voit |
|---|---|
| Jobs | La liste des jobs, un par action ou presque, avec leur durée et leur nombre de stages. |
| Stages | Le découpage de chaque job en stages et le nombre de tasks par stage (1 avant `repartition`, 4 après). |
| Tasks (dans le détail d'un stage) | Une ligne par task : la partition traitée, sa durée, le volume lu. |
| SQL / DataFrame | Le plan de chaque requête sous forme de schéma, le même que celui d'`explain()`. |
| Executors | Les ressources utilisées. En mode local il n'y a qu'un exécuteur, le driver, qui utilise tous les cœurs. |
| Environment | La configuration : version de Java et de Spark, `spark.master = local[*]`, le connecteur MongoDB dans `spark.jars.packages`. |

Les temps affichés ne sont pas représentatifs d'un vrai cluster : en local, avec 800 lignes, le lancement des tasks coûte plus cher que le calcul lui-même.

## Atelier 3 : les indicateurs de BigData Commerce Analytics

On prépare les quatre indicateurs demandés. Deux règles du TP :
- ne jamais additionner `cart_total` ou `cart_discounted_total` sur les lignes de produits, puisqu'ils sont répétés sur chaque ligne du panier ;
- pour le chiffre d'affaires encaissé, on ne garde que les paiements PAID.

On prépare d'abord deux DataFrames de base, sans le champ `_id`, et on les garde en mémoire (`cache`) puisqu'ils servent à tous les indicateurs.

In [7]:
commandes = df_commandes.drop("_id").cache()
paiements = df_payments.drop("_id").cache()
paiements_paid = paiements.filter(F.col("status") == "PAID")

print("Commandes :", commandes.count(), "| paiements :", paiements.count(), "| dont PAID :", paiements_paid.count())

Commandes : 800 | paiements : 208 | dont PAID : 150


### Indicateur 1 : chiffre d'affaires par catégorie

Définition du TP : somme des montants des lignes de produits à partir de `discounted_line_total`. C'est le montant des paniers, payés ou non.

On ajoute une deuxième colonne avec le montant réellement encaissé : on fait une jointure entre les lignes de commandes et les paiements PAID sur `cart_id`, pour ne garder que les lignes des paniers payés.

In [8]:
ca_commande = (
    commandes
    .groupBy("category")
    .agg(F.round(F.sum("discounted_line_total"), 2).alias("ca_commande"))
)

ca_encaisse_cat = (
    commandes
    .join(paiements_paid.select("cart_id"), on="cart_id", how="inner")
    .groupBy("category")
    .agg(F.round(F.sum("discounted_line_total"), 2).alias("ca_encaisse"))
)

total_commande = ca_commande.agg(F.sum("ca_commande")).first()[0]

ind_ca_categorie = (
    ca_commande
    .join(ca_encaisse_cat, on="category", how="left")
    .fillna({"ca_encaisse": 0.0})
    .withColumn("part_ca_commande_pct", F.round(F.col("ca_commande") / total_commande * 100, 1))
    .orderBy(F.desc("ca_commande"))
)
ind_ca_categorie.show(30, truncate=False)

+-------------------+-----------+-----------+--------------------+
|category           |ca_commande|ca_encaisse|part_ca_commande_pct|
+-------------------+-----------+-----------+--------------------+
|vehicle            |1682064.05 |1374964.95 |48.7                |
|motorcycle         |609460.61  |404546.47  |17.6                |
|mens-watches       |593063.86  |446483.7   |17.2                |
|womens-watches     |249189.51  |216806.68  |7.2                 |
|laptops            |85006.24   |51469.78   |2.5                 |
|furniture          |77674.97   |49322.28   |2.2                 |
|smartphones        |73130.95   |62735.15   |2.1                 |
|mobile-accessories |26160.99   |21354.64   |0.8                 |
|tablets            |14890.16   |9098.16    |0.4                 |
|mens-shoes         |8278.9     |6321.99    |0.2                 |
|womens-bags        |5849.91    |2217.36    |0.2                 |
|kitchen-accessories|5714.33    |3837.02    |0.2              

### Indicateur 2 : top 10 des clients

Définition du TP : somme des paiements PAID regroupés par `user_id`.

In [9]:
ind_top_clients = (
    paiements_paid
    .groupBy("user_id")
    .agg(
        F.count("*").alias("nb_paiements"),
        F.round(F.sum("amount"), 2).alias("montant_paye"),
    )
    .orderBy(F.desc("montant_paye"))
    .limit(10)
)
ind_top_clients.show(truncate=False)

total_paid = paiements_paid.agg(F.sum("amount")).first()[0]
total_top10 = ind_top_clients.agg(F.sum("montant_paye")).first()[0]
print(f"Part du top 10 dans le CA encaissé : {total_top10 / total_paid * 100:.1f} %")
print("Nombre de paniers par client :", paiements.groupBy("user_id").count().agg(F.max("count")).first()[0], "au maximum")

+-------+------------+------------+
|user_id|nb_paiements|montant_paye|
+-------+------------+------------+
|30     |1           |155507.16   |
|52     |1           |130175.28   |
|95     |1           |129783.31   |
|83     |1           |125679.0    |
|33     |1           |125461.58   |
|18     |1           |124978.81   |
|45     |1           |121385.95   |
|155    |1           |110543.89   |
|78     |1           |109890.08   |
|27     |1           |100496.36   |
+-------+------------+------------+

Part du top 10 dans le CA encaissé : 46.2 %
Nombre de paniers par client : 1 au maximum


### Indicateur 3 : taux de refus

Définition du TP : nombre de paiements REFUSED / nombre total de tentatives.

In [10]:
nb_total = paiements.count()

ind_statuts = (
    paiements
    .groupBy("status")
    .agg(
        F.count("*").alias("nb_paiements"),
        F.round(F.sum("amount"), 2).alias("montant"),
    )
    .withColumn("part_pct", F.round(F.col("nb_paiements") / nb_total * 100, 2))
    .orderBy(F.desc("nb_paiements"))
)
ind_statuts.show()

nb_refused = paiements.filter(F.col("status") == "REFUSED").count()
taux_refus = nb_refused / nb_total * 100
print(f"Taux de refus : {nb_refused} / {nb_total} = {taux_refus:.2f} %")

+-------+------------+----------+--------+
| status|nb_paiements|   montant|part_pct|
+-------+------------+----------+--------+
|   PAID|         150|2669235.61|   72.12|
|REFUSED|          37| 423555.75|   17.79|
|PENDING|          21| 363918.22|    10.1|
+-------+------------+----------+--------+

Taux de refus : 37 / 208 = 17.79 %


### Indicateur 4 : évolution mensuelle

Définition du TP : montants des paiements PAID regroupés par mois. Le mois vient de `payment_date`, elle-même tirée de `order_date`, qui est une date générée pour l'exercice.

In [11]:
ind_mensuel = (
    paiements_paid
    .withColumn("mois", F.date_format("payment_date", "yyyy-MM"))
    .groupBy("mois")
    .agg(
        F.count("*").alias("nb_paiements"),
        F.round(F.sum("amount"), 2).alias("ca_encaisse"),
    )
    .orderBy("mois")
)
ind_mensuel.show(12)

+-------+------------+-----------+
|   mois|nb_paiements|ca_encaisse|
+-------+------------+-----------+
|2025-01|          13|   74046.27|
|2025-02|          13|  432281.84|
|2025-03|          16|  384654.44|
|2025-04|           9|  228117.54|
|2025-05|           5|   128046.3|
|2025-06|          12|  321302.66|
|2025-07|          15|   54081.45|
|2025-08|          10|  140272.22|
|2025-09|          13|  102087.86|
|2025-10|          15|  203587.26|
|2025-11|          11|  285127.81|
|2025-12|          18|  315629.96|
+-------+------------+-----------+



### Sauvegarde des indicateurs pour le jour 3

Les résultats sont petits (quelques dizaines de lignes), on les ramène dans pandas avec `toPandas()` et on les enregistre en CSV dans `resultats/` à la racine du projet. Le jour 3 pourra les relire directement pour les graphiques. On passe par pandas plutôt que par `df.write` de Spark, qui sous Windows a besoin d'outils Hadoop supplémentaires.

In [12]:
racine = Path.cwd()
while racine != racine.parent and not (racine / "data").exists():
    racine = racine.parent
dossier_resultats = racine / "resultats"
dossier_resultats.mkdir(exist_ok=True)

indicateurs = {
    "ca_par_categorie": ind_ca_categorie,
    "top10_clients": ind_top_clients,
    "repartition_statuts": ind_statuts,
    "evolution_mensuelle": ind_mensuel,
}
for nom, df in indicateurs.items():
    chemin = dossier_resultats / f"{nom}.csv"
    df.toPandas().to_csv(chemin, index=False)
    print("Enregistré :", chemin.relative_to(racine))

Enregistré : resultats\ca_par_categorie.csv
Enregistré : resultats\top10_clients.csv
Enregistré : resultats\repartition_statuts.csv
Enregistré : resultats\evolution_mensuelle.csv


### Commentaire des résultats

**Indicateur 1, chiffre d'affaires par catégorie.** Le montant total des paniers est de 3 456 709,58, dont 2 669 235,61 encaissés. Le chiffre d'affaires est très concentré : `vehicle` représente à elle seule 48,7 % du montant des paniers (1 682 064,05), et avec `motorcycle`, `mens-watches` et `womens-watches` on arrive à environ 91 %. La plupart des autres catégories pèsent moins de 1 %. Ce classement reflète surtout le prix des produits (une voiture à plusieurs dizaines de milliers contre des produits à quelques euros) plus que le nombre d'articles vendus.

**Indicateur 2, top 10 des clients.** Le premier client (`user_id` 30) a payé 155 507,16 et les dix premiers représentent 46,2 % du chiffre d'affaires encaissé, soit près de la moitié pour 10 clients sur 150 ayant payé. Attention à l'interprétation : chaque client n'a qu'un seul panier dans les données DummyJSON, donc ce top 10 est en fait le top 10 des paniers payés les plus chers. On ne peut rien dire sur la fidélité des clients.

**Indicateur 3, taux de refus.** 37 paiements refusés sur 208 tentatives, soit 17,79 %, un peu en dessous des 20 % prévus par la simulation (effet du tirage au hasard sur 208 paniers). Ces refus représentent 423 555,75 de commandes non encaissées.

**Indicateur 4, évolution mensuelle.** Le nombre de paiements PAID varie peu d'un mois à l'autre (entre 5 et 18), mais les montants varient énormément : 54 081,45 en juillet contre 432 281,84 en février. Ces écarts viennent de quelques paniers très chers (véhicules) qui tombent un mois plutôt qu'un autre, pas d'une hausse du nombre de ventes. Comme les dates sont générées pour l'exercice, on ne peut pas en tirer de conclusion sur une saisonnalité réelle.

Ces quatre fichiers (`resultats/*.csv`) serviront de base aux graphiques du jour 3.